# Exploratory Data Analysis — 30-Day Readmission Model

Reconstructed EDA notebook for the hospital readmission prediction project.
Pulls directly from the `features` schema (`cohort`, `utilization_features`,
`medication_features`, `condition_features`, `model_features`) in the project's
RDS PostgreSQL database.

**Goal:** understand the population, the target class balance, missingness
patterns, and how each feature domain relates to 30-day readmission before
any modeling decisions are made.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sqlalchemy import create_engine

pd.set_option('display.max_columns', 50)
sns.set_theme(style='whitegrid')
%matplotlib inline

## 1. Connect to the feature store

Credentials are pulled from environment variables. Set these in your shell/`.env` before running:

```
export PGHOST=your-rds-endpoint
export PGPORT=5432
export PGDATABASE=readmission
export PGUSER=your_user
export PGPASSWORD=your_password
```

In [ ]:
PGHOST = os.environ.get("PGHOST", "localhost")
PGPORT = os.environ.get("PGPORT", "5432")
PGDATABASE = os.environ.get("PGDATABASE", "readmission")
PGUSER = os.environ.get("PGUSER", "postgres")
PGPASSWORD = os.environ.get("PGPASSWORD", "")

engine = create_engine(
    f"postgresql+psycopg2://{PGUSER}:{PGPASSWORD}@{PGHOST}:{PGPORT}/{PGDATABASE}"
)

with engine.connect() as conn:
    print("Connected:", conn.engine.url.render_as_string(hide_password=True))

## 2. Load the feature tables

In [ ]:
cohort = pd.read_sql("SELECT * FROM features.cohort", engine)
utilization = pd.read_sql("SELECT * FROM features.utilization_features", engine)
medication = pd.read_sql("SELECT * FROM features.medication_features", engine)
condition = pd.read_sql("SELECT * FROM features.condition_features", engine)
model_features = pd.read_sql("SELECT * FROM features.model_features", engine)

for name, df in [("cohort", cohort), ("utilization", utilization),
                  ("medication", medication), ("condition", condition),
                  ("model_features", model_features)]:
    print(f"{name:15s} shape={df.shape}")

## 3. Cohort overview

Row-level shape, dtypes, and basic descriptive stats for the base population.

In [ ]:
cohort.dtypes

In [ ]:
cohort.describe(include='all').T

## 4. Target distribution — class imbalance check

`readmit_30d` is expected to be imbalanced (readmission is the minority
class). This directly informs the choice of AUC-PR as the primary
evaluation metric later in modeling.

In [ ]:
target_counts = cohort['readmit_30d'].value_counts(dropna=False)
target_rate = cohort['readmit_30d'].mean()

print(target_counts)
print(f"\nPositive class rate: {target_rate:.2%}")

fig, ax = plt.subplots(figsize=(5, 4))
target_counts.sort_index().plot(kind='bar', ax=ax, color=['#4FD1C5', '#E8A33D'])
ax.set_xticklabels(['No Readmission (0)', 'Readmission (1)'], rotation=0)
ax.set_ylabel('Stay count')
ax.set_title('30-Day Readmission Class Balance')
plt.tight_layout()
plt.show()

## 5. Missingness overview

Missingness is treated as MNAR (missing-not-at-random) in this project. The
fact that a value is missing (e.g., no A1c on file) can itself be
informative, so this is a diagnostic step, not just a cleaning step.

In [ ]:
missing_pct = (
    model_features.isna().mean()
    .sort_values(ascending=False)
    .rename('pct_missing')
    .to_frame()
)
missing_pct = missing_pct[missing_pct['pct_missing'] > 0]

display(missing_pct)

if not missing_pct.empty:
    fig, ax = plt.subplots(figsize=(7, 4))
    missing_pct['pct_missing'].plot(kind='barh', ax=ax, color='#4FD1C5')
    ax.set_xlabel('% missing')
    ax.set_title('Missingness by Feature (features.model_features)')
    plt.tight_layout()
    plt.show()
else:
    print("No missing values detected in model_features.")

## 6. Demographics — age & gender

Basic population shape. `gender_male` / `gender_female` are stored as
separate binary flags in `cohort`.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

sns.histplot(cohort['age'].dropna(), bins=30, ax=axes[0], color='#4FD1C5')
axes[0].set_title('Age Distribution')
axes[0].set_xlabel('Age')

gender_counts = pd.Series({
    'Male': cohort['gender_male'].sum(),
    'Female': cohort['gender_female'].sum(),
})
gender_counts.plot(kind='bar', ax=axes[1], color=['#4FD1C5', '#E8A33D'])
axes[1].set_title('Gender Breakdown')
axes[1].set_xticklabels(gender_counts.index, rotation=0)

plt.tight_layout()
plt.show()

## 7. Length of stay & cost

Look at `los` and `total_stay_cost` overall, and split by readmission
outcome. A longer or costlier index stay is a plausible readmission
signal worth checking early.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

sns.boxplot(data=cohort, x='readmit_30d', y='los', ax=axes[0], palette=['#4FD1C5', '#E8A33D'])
axes[0].set_title('Length of Stay by Readmission Outcome')
axes[0].set_xticklabels(['No Readmit', 'Readmit'])

sns.boxplot(data=cohort, x='readmit_30d', y='total_stay_cost', ax=axes[1], palette=['#4FD1C5', '#E8A33D'])
axes[1].set_title('Total Stay Cost by Readmission Outcome')
axes[1].set_xticklabels(['No Readmit', 'Readmit'])

plt.tight_layout()
plt.show()

## 8. Utilization features

Prior admissions and ED visits are the classic "recent healthcare
utilization predicts future utilization" features. Compare distributions
by outcome.

In [ ]:
util_cohort = utilization.merge(cohort[['stay_id', 'readmit_30d']], on='stay_id', how='left')

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

sns.boxplot(data=util_cohort, x='readmit_30d', y='prior_admissions_180d', ax=axes[0], palette=['#4FD1C5', '#E8A33D'])
axes[0].set_title('Prior Admissions (180d) by Outcome')
axes[0].set_xticklabels(['No Readmit', 'Readmit'])

sns.boxplot(data=util_cohort, x='readmit_30d', y='ed_visits_180d', ax=axes[1], palette=['#4FD1C5', '#E8A33D'])
axes[1].set_title('ED Visits (180d) by Outcome')
axes[1].set_xticklabels(['No Readmit', 'Readmit'])

plt.tight_layout()
plt.show()

util_cohort.groupby('readmit_30d')[
    ['prior_admissions_90d', 'prior_admissions_180d', 'prior_admissions_365d',
     'ed_visits_90d', 'ed_visits_180d', 'ed_visits_365d', 'days_since_last_admit']
].mean()

## 9. Condition & medication burden

Chronic condition count, diabetes/heart failure prevalence, latest A1c,
and active medication count.

In [ ]:
cond_cohort = condition.merge(cohort[['stay_id', 'readmit_30d']], on='stay_id', how='left')
med_cohort = medication.merge(cohort[['stay_id', 'readmit_30d']], on='stay_id', how='left')

fig, axes = plt.subplots(2, 2, figsize=(11, 8))

sns.boxplot(data=cond_cohort, x='readmit_30d', y='active_conditions', ax=axes[0, 0], palette=['#4FD1C5', '#E8A33D'])
axes[0, 0].set_title('Active Conditions by Outcome')
axes[0, 0].set_xticklabels(['No Readmit', 'Readmit'])

sns.boxplot(data=cond_cohort, x='readmit_30d', y='latest_a1c', ax=axes[0, 1], palette=['#4FD1C5', '#E8A33D'])
axes[0, 1].set_title('Latest A1c by Outcome')
axes[0, 1].set_xticklabels(['No Readmit', 'Readmit'])

prevalence = cond_cohort.groupby('readmit_30d')[['has_diabetes', 'has_heart_failure']].mean()
prevalence.T.plot(kind='bar', ax=axes[1, 0], color=['#4FD1C5', '#E8A33D'])
axes[1, 0].set_title('Condition Prevalence by Outcome')
axes[1, 0].set_xticklabels(['Diabetes', 'Heart Failure'], rotation=0)
axes[1, 0].legend(['No Readmit', 'Readmit'])

sns.boxplot(data=med_cohort, x='readmit_30d', y='active_medications', ax=axes[1, 1], palette=['#4FD1C5', '#E8A33D'])
axes[1, 1].set_title('Active Medications by Outcome')
axes[1, 1].set_xticklabels(['No Readmit', 'Readmit'])

plt.tight_layout()
plt.show()

## 10. Correlation with the target

Point-biserial correlation of each numeric feature against the binary
`readmit_30d` target, sorted to surface the strongest univariate signals.

In [ ]:
numeric_cols = model_features.select_dtypes(include=[np.number]).columns.drop('readmit_30d')

corrs = (
    model_features[numeric_cols.tolist() + ['readmit_30d']]
    .corr()['readmit_30d']
    .drop('readmit_30d')
    .sort_values(key=abs, ascending=False)
)

fig, ax = plt.subplots(figsize=(7, 5))
corrs.plot(kind='barh', ax=ax, color='#4FD1C5')
ax.set_title('Feature Correlation with 30-Day Readmission')
ax.set_xlabel('Correlation coefficient')
plt.tight_layout()
plt.show()

corrs

## 11. Feature-to-feature correlation (multicollinearity check)

Useful mainly for the logistic regression baseline. Tree-based models
(XGBoost, random forest) are largely robust to correlated features, but
it's worth knowing where the redundancy sits.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
corr_matrix = model_features[numeric_cols].corr()
sns.heatmap(corr_matrix, cmap='RdBu_r', center=0, annot=False, ax=ax)
ax.set_title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()

## 12. Key EDA takeaways

*Fill this in after re-running against live data.*

- Class balance: `readmit_30d` positive rate ≈ **[fill in]**
- Strongest univariate correlates with the target: **[fill in]**
- Notable missingness: **[fill in]**
- Multicollinearity flags: **[fill in]**